# dbscoring Polars Lab

Полностью исполняемая лабораторная работа на `polars`: физическая модель, warehouse, процессы обновления, тестовый ML scoring pipeline и примеры пользовательского взаимодействия.

## 1. Setup

Ноутбук использует тот же production package, что и CLI и тесты. Это важно: код не дублируется, а демонстрируется через стабильные функции пакета.

In [ ]:
from pathlib import Path

from dbscoring.etl import (
    build_feature_frame,
    build_warehouse,
    read_table,
    validate_warehouse,
)
from dbscoring.ml import make_synthetic_labels, predict, train_catboost, tune_catboost
from dbscoring.paths import ProjectPaths

paths = ProjectPaths(
    data_root=Path("../data/sources"), warehouse_root=Path("../data/warehouse")
).resolve()

## 2. Build warehouse

Full refresh читает реальные parquet partitions, строит справочники, нормализует monthly SCD1 и daily SCD2 атрибуты и пишет `load_log`.

In [ ]:
summary = build_warehouse(paths, reset=True)
summary.as_dict()

## 3. Validate warehouse

Проверяются схемы, первичные ключи, статусы `load_log`, размерность справочников и таблиц атрибутов.

In [ ]:
validation_summary = validate_warehouse(paths)
validation_summary.as_dict()

## 4. User-facing warehouse examples

Эти команды повторяют то же взаимодействие через CLI/TUI:

```bash
uv run dbscoring status
uv run dbscoring warehouse build --data-root data/sources --warehouse-root data/warehouse
uv run dbscoring warehouse validate --warehouse-root data/warehouse
uv run dbscoring warehouse report --warehouse-root data/warehouse
```


In [ ]:
load_log = read_table(paths, "load_log")
load_log.group_by(["target_table", "load_status"]).len().sort(
    ["target_table", "load_status"]
)

## 5. Build ML features and demo labels

В текущих данных нет реального бизнес-target. Поэтому production API ожидает внешний label dataset, а эта ячейка создаёт детерминированные synthetic labels только для демонстрации и тестов.

In [ ]:
features = build_feature_frame(paths)
labels = make_synthetic_labels(features)
features.head(), labels.head()

## 6. Train and predict

Минимальный CatBoost-прогон показывает полный путь обучения и инференса. Для production-обучения замените `labels` на реальный label provider.

In [ ]:
model = train_catboost(features, labels, iterations=20, depth=3, learning_rate=0.08)
predictions = predict(model, features)
model.metrics, predictions.head()

## 7. Optuna tuning example

Небольшой trial budget используется для интерактивного запуска. В production увеличьте `trials`.

In [ ]:
tuning_result = tune_catboost(features, labels, trials=3)
tuning_result